# Test the receptive-field p-value filter (p_value_rf < 0.01)

Per Siegle et al. 2021 (Allen Neuropixels Visual Coding) Extended Data Fig. 4:
> *Note that we do not use the default AllenSDK filters in this work, but instead use a receptive field P value of 0.01 as the primary metric for selecting units for analysis.*

We already have all the data we need on disk:
- `data/unit_analysis_metrics.pkl` — full per-unit metrics including `p_value_rf` (no AllenSDK QC pre-filter applied).
- `spike_times_v2/<session_id>_alllayers_spiketimes.pkl` — VISp spike times for every unit, no QC pre-filter.

So we can apply the RF p-value criterion purely as a post-hoc lookup — no re-downloads.

This notebook:
1. Loads the metrics and inspects `p_value_rf` (coverage, distribution).
2. Applies `p_value_rf < 0.01` and reports passing counts overall + per session + per area.
3. Cross-references against the VISp spike-time pickles to confirm how many VISp units survive the cut.
4. Visualises the p_value_rf distribution and the AllenSDK default QC thresholds for comparison.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT  = Path('..').resolve()
METRICS    = REPO_ROOT / 'data' / 'unit_analysis_metrics.pkl'
CACHE_DIR  = Path('/Users/pmccarthy/Documents/experimental_data/allen_visual_neuropixels_longwindow_5ms_bins')
SPIKES_DIR = CACHE_DIR / 'spike_times_v2'

P_RF_THRESHOLD = 0.01

print(f'Metrics file: {METRICS}  (exists={METRICS.exists()})')
print(f'Spike-times dir: {SPIKES_DIR}  (exists={SPIKES_DIR.exists()})')

In [ ]:
with open(METRICS, 'rb') as f:
    metrics = pickle.load(f)

# Flatten {session_id: {unit_id: {col: val}}} into a long DataFrame
frames = []
for sid, units in metrics.items():
    df = pd.DataFrame.from_dict(units, orient='index')
    df.index.name = 'unit_id'
    df['session_id'] = sid
    frames.append(df.reset_index())
all_units = pd.concat(frames, ignore_index=True)

print(f'Sessions: {all_units.session_id.nunique()}')
print(f'Units:    {len(all_units):,}')
print(f'Columns containing "rf": {[c for c in all_units.columns if "rf" in c.lower()]}')

## 1. Coverage of `p_value_rf`

Units recorded in sessions that didn't run the gabors RF-mapping stimulus, or with no measurable RF, will have `NaN`. The `< 0.01` comparison drops those automatically, which is what we want.

In [ ]:
n_total    = len(all_units)
n_notna    = all_units['p_value_rf'].notna().sum()
n_pass     = (all_units['p_value_rf'] < P_RF_THRESHOLD).sum()

print(f'Total units:                       {n_total:,}')
print(f'With non-NaN p_value_rf:           {n_notna:,}  ({100*n_notna/n_total:.1f}%)')
print(f'Passing p_value_rf < {P_RF_THRESHOLD}:           {n_pass:,}  ({100*n_pass/n_total:.1f}%)')
print()
print('Summary of p_value_rf (non-NaN):')
print(all_units['p_value_rf'].dropna().describe().to_string())

In [ ]:
by_session = (
    all_units
    .assign(passes=lambda d: d['p_value_rf'] < P_RF_THRESHOLD)
    .groupby('session_id')
    .agg(total=('unit_id', 'size'),
         passes=('passes', 'sum'))
    .assign(frac=lambda d: d['passes'] / d['total'])
    .sort_index()
)
print(by_session.to_string())
print(f'\nMean pass rate across sessions: {by_session.frac.mean():.1%}')

## 2. Pass rate by brain area

RF mapping primarily targets visually-driven units, so visual cortex should pass at a much higher rate than thalamus/hippocampus etc.

In [ ]:
by_area = (
    all_units
    .assign(passes=lambda d: d['p_value_rf'] < P_RF_THRESHOLD)
    .groupby('ecephys_structure_acronym', dropna=False)
    .agg(total=('unit_id', 'size'),
         passes=('passes', 'sum'))
    .assign(frac=lambda d: d['passes'] / d['total'])
    .sort_values('total', ascending=False)
)
print(by_area.head(20).to_string())

## 3. Cross-reference with the extracted VISp spike-time pickles

How many of the VISp units we've already extracted to disk would survive the cut?

In [ ]:
rows = []
pkl_files = sorted(SPIKES_DIR.glob('*_alllayers_spiketimes.pkl')) if SPIKES_DIR.exists() else []
if not pkl_files:
    print(f'No spike-time pickles found at {SPIKES_DIR}')

for pkl in pkl_files:
    sid = int(pkl.stem.split('_')[0])
    with open(pkl, 'rb') as f:
        d = pickle.load(f)
    visp_ids = np.asarray(d['unit_ids']).astype(int)

    sess_m = metrics.get(sid, {})
    sess_df = pd.DataFrame.from_dict(sess_m, orient='index')
    sess_df.index = sess_df.index.astype(int)

    in_metrics = sess_df.loc[sess_df.index.intersection(visp_ids)]
    passing    = in_metrics[in_metrics['p_value_rf'] < P_RF_THRESHOLD]
    rows.append({
        'session_id':     sid,
        'visp_units':     len(visp_ids),
        'in_metrics':     len(in_metrics),
        'rf_notna':       in_metrics['p_value_rf'].notna().sum(),
        'pass_rf_lt_001': len(passing),
    })

visp_summary = pd.DataFrame(rows).set_index('session_id')
print(visp_summary.to_string())
if len(visp_summary):
    print(f'\nTotal VISp units extracted: {visp_summary.visp_units.sum():,}')
    print(f'Total passing RF filter:    {visp_summary.pass_rf_lt_001.sum():,} '
          f'({100*visp_summary.pass_rf_lt_001.sum()/max(visp_summary.visp_units.sum(),1):.1f}%)')

## 4. Distribution plots

Left: `p_value_rf` distribution with the 0.01 cutoff marked. Right: the AllenSDK default QC metrics we are *not* using as primary filters (per the manuscript).

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

p = all_units['p_value_rf'].dropna()
axes[0].hist(p, bins=50, color='#4c72b0', edgecolor='white')
axes[0].axvline(P_RF_THRESHOLD, color='crimson', linestyle='--', label=f'{P_RF_THRESHOLD}')
axes[0].set_xlabel('p_value_rf')
axes[0].set_ylabel('units')
axes[0].set_title('Receptive-field p-value')
axes[0].legend()

axes[1].hist(all_units['amplitude_cutoff'].dropna(), bins=50, color='#55a868', edgecolor='white')
axes[1].axvline(0.1, color='crimson', linestyle='--', label='SDK default 0.1')
axes[1].set_xlabel('amplitude_cutoff')
axes[1].set_title('Amplitude cutoff')
axes[1].legend()

axes[2].hist(all_units['presence_ratio'].dropna(), bins=50, color='#c44e52', edgecolor='white')
axes[2].axvline(0.9, color='crimson', linestyle='--', label='SDK default 0.9')
axes[2].set_xlabel('presence_ratio')
axes[2].set_title('Presence ratio')
axes[2].legend()

axes[3].hist(all_units['isi_violations'].dropna().clip(upper=2), bins=50, color='#8172b2', edgecolor='white')
axes[3].axvline(0.5, color='crimson', linestyle='--', label='SDK default 0.5')
axes[3].set_xlabel('isi_violations (clipped at 2)')
axes[3].set_title('ISI violations')
axes[3].legend()

plt.tight_layout()
plt.show()

## 5. Overlap with default AllenSDK QC

Just so you can see what you're trading off — how many units survive the RF filter alone vs. the SDK default QC alone vs. both.

In [ ]:
u = all_units
pass_rf  = u['p_value_rf'] < P_RF_THRESHOLD
pass_qc  = (u['amplitude_cutoff'] < 0.1) & (u['presence_ratio'] > 0.9) & (u['isi_violations'] < 0.5)

print(f'RF filter only      : {pass_rf.sum():>7,}')
print(f'SDK QC only         : {pass_qc.sum():>7,}')
print(f'RF AND SDK QC       : {(pass_rf & pass_qc).sum():>7,}')
print(f'RF only, fails QC   : {(pass_rf & ~pass_qc).sum():>7,}')
print(f'Passes QC, fails RF : {(~pass_rf & pass_qc).sum():>7,}')

## 6. Test the filter-mask script

Once you've run `scripts/20_05_26_filter_units_rf_pvalue.py`, this loads its output and double-checks it matches the in-notebook computation.

In [ ]:
mask_path = REPO_ROOT / 'data' / 'units_rf_pvalue_lt_001.pkl'
if not mask_path.exists():
    print(f'Mask not found at {mask_path}. Run scripts/20_05_26_filter_units_rf_pvalue.py first.')
else:
    with open(mask_path, 'rb') as f:
        mask = pickle.load(f)
    print(f'Mask covers {len(mask["passing_unit_ids"])} sessions')
    print(f'Threshold:  {mask["threshold"]}')
    print(f'Total passing units: {sum(len(v) for v in mask["passing_unit_ids"].values()):,}')

    # Spot-check first session matches in-notebook computation
    sid = next(iter(mask['passing_unit_ids']))
    from_mask  = set(int(u) for u in mask['passing_unit_ids'][sid])
    sess_df = pd.DataFrame.from_dict(metrics[sid], orient='index')
    sess_df.index = sess_df.index.astype(int)
    from_notebook = set(sess_df.index[sess_df['p_value_rf'] < P_RF_THRESHOLD])
    assert from_mask == from_notebook, (
        f'Mismatch for session {sid}: '
        f'mask has {len(from_mask)}, notebook has {len(from_notebook)}'
    )
    print(f'\nSession {sid}: mask and in-notebook computation agree ({len(from_mask)} units).')